In [2]:
from transformers import RobertaTokenizer, DataCollatorWithPadding
import numpy as np
import pandas as pd
from torch.utils.data import DataLoader
from tqdm import tqdm
from peft import AutoPeftModelForSequenceClassification
import torch
from datasets import load_from_disk
from sklearn.metrics import accuracy_score, confusion_matrix, precision_recall_fscore_support

/Users/ngokykhanhthu/2026/chewy/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1028 20:33:21.066000 2565 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


In [3]:
DATA_PATH = "./../data/"
ROBERTA_CONFIGS_PATHS = DATA_PATH + "roberta_configs/"
FROZEN_TEST_PATH = ROBERTA_CONFIGS_PATHS + "test_dataset.csv"
ROBERTA_WEIGHTS_PATH = ROBERTA_CONFIGS_PATHS + "peft_model"
TOKENIZER_PATH = ROBERTA_CONFIGS_PATHS + "tokenizer"
TOKENIZED_TEST_PATH = ROBERTA_CONFIGS_PATHS + "tokenized_test"
id2label = {0: 'human', 1: 'LLM-generated'}


# stubbin the streamlined evals pipeline

In [4]:
inference_model = AutoPeftModelForSequenceClassification.from_pretrained(ROBERTA_WEIGHTS_PATH, id2label=id2label)

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 4817.55it/s]
[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.dense.bias         | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [5]:
frozen_test = pd.read_csv(FROZEN_TEST_PATH)

In [6]:
saved_tokenizer = RobertaTokenizer.from_pretrained(TOKENIZER_PATH)
data_collator = DataCollatorWithPadding(tokenizer=saved_tokenizer, return_tensors="pt")

In [ ]:
tokenized_test_dataset = load_from_disk(TOKENIZED_TEST_PATH)